<a href="https://colab.research.google.com/github/laboratoriodecodigos/Colab-Python/blob/main/SQL_Machine_Learning_Titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# IMPORTAR LIBRERÍAS Y CARGAR TITANIC
import sqlite3
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [8]:
# Cargar dataset Titanic de Seaborn
titanic = sns.load_dataset("titanic")

titanic

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True
888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True


In [9]:
# Cargar dataset Titanic de Seaborn
titanic = sns.load_dataset("titanic")

# Dejando columnas esenciales
titanic = titanic[["survived","pclass","sex",
                   "age","sibsp","parch","fare","embarked"]]

In [10]:
# Crear DB SQLite
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

# Guardar el dataset en SQL
titanic.to_sql("titanic", conn, index=False, if_exists="replace")

891

In [11]:
titanic

,survived,pclass,sex,age,sibsp,parch,fare,embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S
...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S
887,1,1,female,19.0,0,0,30.0000,S
888,0,3,female,NaN,1,2,23.4500,S
889,1,1,male,26.0,0,0,30.0000,C


In [12]:
# SQL BÁSICO: VER REGISTROS, FILTRAR Y ORDENAR
print("\nPrimeras filas:")
display(pd.read_sql_query("SELECT * FROM titanic LIMIT 5", conn))


Primeras filas:


,survived,pclass,sex,age,sibsp,parch,fare,embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [13]:
print("\nPasajeros con edad > 50:")
display(pd.read_sql_query("SELECT * FROM titanic WHERE age > 50 LIMIT 5", conn))


Pasajeros con edad > 50:


,survived,pclass,sex,age,sibsp,parch,fare,embarked
0,0,1,male,54.0,0,0,51.8625,S
1,1,1,female,58.0,0,0,26.5500,S
2,1,2,female,55.0,0,0,16.0000,S
3,0,2,male,66.0,0,0,10.5000,S
4,0,1,male,65.0,0,1,61.9792,C


In [14]:
print("\nTop 5 tarifas más altas:")
display(pd.read_sql_query("SELECT * FROM titanic ORDER BY fare DESC LIMIT 5", conn))


Top 5 tarifas más altas:


,survived,pclass,sex,age,sibsp,parch,fare,embarked
0,1,1,female,35.0,0,0,512.3292,C
1,1,1,male,36.0,0,1,512.3292,C
2,1,1,male,35.0,0,0,512.3292,C
3,0,1,male,19.0,3,2,263.0000,S
4,1,1,female,23.0,3,2,263.0000,S


In [15]:
print("\nSupervivencia por clase:")
display(pd.read_sql_query("""
SELECT pclass, AVG(survived) AS tasa_supervivencia
FROM titanic
GROUP BY pclass
""", conn))


Supervivencia por clase:


,pclass,tasa_supervivencia
0,1,0.629630
1,2,0.472826
2,3,0.242363


In [16]:
print("\nSupervivencia por sexo:")
display(pd.read_sql_query("""
SELECT sex, AVG(survived) AS tasa_supervivencia
FROM titanic
GROUP BY sex
""", conn))


Supervivencia por sexo:


,sex,tasa_supervivencia
0,female,0.742038
1,male,0.188908


In [17]:
print("\nCTE: edad promedio por clase:")
display(pd.read_sql_query("""
WITH promedio AS (
    SELECT pclass, AVG(age) AS edad_promedio
    FROM titanic
    GROUP BY pclass
)
SELECT t.pclass, t.age, p.edad_promedio
FROM titanic t
JOIN promedio p ON t.pclass = p.pclass
LIMIT 10
""", conn))



CTE: edad promedio por clase:


,pclass,age,edad_promedio
0,3,22.0,25.140620
1,1,38.0,38.233441
2,3,26.0,25.140620
3,1,35.0,38.233441
4,3,35.0,25.140620
5,3,NaN,25.140620
6,1,54.0,38.233441
7,3,2.0,25.140620
8,3,27.0,25.140620
9,2,14.0,29.877630


In [18]:
print("\nRanking de tarifas dentro de cada clase:")
display(pd.read_sql_query("""
SELECT
    pclass,
    fare,
    age,
    RANK() OVER (PARTITION BY pclass ORDER BY fare DESC) AS ranking_fare
FROM titanic
WHERE fare IS NOT NULL
LIMIT 10
""", conn))


Ranking de tarifas dentro de cada clase:


,pclass,fare,age,ranking_fare
0,1,512.3292,35.0,1
1,1,512.3292,36.0,1
2,1,512.3292,35.0,1
3,1,263.0000,19.0,4
4,1,263.0000,23.0,4
5,1,263.0000,24.0,4
6,1,263.0000,64.0,4
7,1,262.3750,18.0,8
8,1,262.3750,21.0,8
9,1,247.5208,24.0,10


In [19]:
# Reemplazar valores nulos con SQL
cursor.execute("""
CREATE TABLE titanic_clean AS
SELECT
    survived,
    pclass,
    CASE WHEN sex='male' THEN 0 ELSE 1 END AS sex,
    COALESCE(age, 30) AS age,       -- Relleno simple
    sibsp,
    parch,
    COALESCE(fare, (SELECT AVG(fare) FROM titanic)) AS fare,
    CASE
        WHEN embarked='S' THEN 0
        WHEN embarked='C' THEN 1
        WHEN embarked='Q' THEN 2
        ELSE 0
    END AS embarked
FROM titanic
""")

In [20]:
titanic_clean = pd.read_sql_query("SELECT * FROM titanic_clean", conn)
titanic_clean
#

,survived,pclass,sex,age,sibsp,parch,fare,embarked
0,0,3,0,22.0,1,0,7.2500,0
1,1,1,1,38.0,1,0,71.2833,1
2,1,3,1,26.0,0,0,7.9250,0
3,1,1,1,35.0,1,0,53.1000,0
4,0,3,0,35.0,0,0,8.0500,0
...,...,...,...,...,...,...,...,...
886,0,2,0,27.0,0,0,13.0000,0
887,1,1,1,19.0,0,0,30.0000,0
888,0,3,1,30.0,1,2,23.4500,0
889,1,1,0,26.0,0,0,30.0000,1


In [21]:
df = pd.read_sql_query("SELECT * FROM titanic_clean", conn)

X = df.drop("survived", axis=1)
y = df["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

model = DecisionTreeClassifier(max_depth=4)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print("\nPrecisión del modelo:", acc)


Precisión del modelo: 0.8071748878923767
